In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Find the project folder
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "employee.csv").exists():
        break
    if (PROJECT_ROOT / "data" / "employee.csv").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

# Check dataset location
if (PROJECT_ROOT / "employee.csv").exists():
    DATA_PATH = PROJECT_ROOT / "employee.csv"
else:
    DATA_PATH = PROJECT_ROOT / "data" / "employee.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
# Feature 1: Employee tenure group
df["TenureGroup"] = pd.cut(
    df["YearsAtCompany"],
    bins=[-1, 1, 5, 10, 100],
    labels=["New", "Short", "Medium", "Long"]
)

# Feature 2: Income per working year
df["IncomePerWorkingYear"] = (
    df["MonthlyIncome"] / (df["TotalWorkingYears"] + 1)
)

# Feature 3: Distance group
df["DistanceGroup"] = pd.cut(
    df["DistanceFromHome"],
    bins=[0, 5, 15, 100],
    labels=["Near", "Medium", "Far"]
)

# Feature 4: Promotion gap group
df["PromotionGapGroup"] = pd.cut(
    df["YearsSinceLastPromotion"],
    bins=[-1, 1, 3, 100],
    labels=["Recent", "Moderate", "Long"]
)

print("Four new features created successfully!")

df[[
    "TenureGroup",
    "IncomePerWorkingYear",
    "DistanceGroup",
    "PromotionGapGroup"
]].head()

Four new features created successfully!


,TenureGroup,IncomePerWorkingYear,DistanceGroup,PromotionGapGroup
0,Medium,665.888889,Near,Recent
1,Medium,466.363636,Medium,Recent
2,New,261.250000,Near,Recent
3,Medium,323.222222,Near,Moderate
4,Short,495.428571,Near,Moderate


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Remove unnecessary columns
drop_columns = [
    "Attrition",
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

X = df.drop(columns=drop_columns, errors="ignore")

# Convert target into numbers
y = df["Attrition"].map({"Yes": 1, "No": 0})

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Data split completed successfully!")

Training data: (1176, 34)
Testing data: (294, 34)
Data split completed successfully!


In [7]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical columns:", len(numeric_features))
print("Categorical columns:", len(categorical_features))

Numerical columns: 24
Categorical columns: 10


C:\Users\Admin\AppData\Local\Temp\ipykernel_14392\395403489.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("ColumnTransformer created successfully!")

ColumnTransformer created successfully!


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
import pandas as pd

In [10]:
logistic_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

print("Logistic Regression pipeline created!")

Logistic Regression pipeline created!


In [11]:
random_forest_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42
    ))
])

print("Random Forest pipeline created!")

Random Forest pipeline created!


In [12]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "roc_auc": "roc_auc",
    "f1": "f1"
}

print("5-fold cross-validation is ready!")

5-fold cross-validation is ready!


In [13]:
logistic_scores = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Logistic Regression Results")
print("Mean ROC-AUC:", logistic_scores["test_roc_auc"].mean())
print("Mean F1:", logistic_scores["test_f1"].mean())

Logistic Regression Results
Mean ROC-AUC: 0.8298476347768359
Mean F1: 0.5071870556884719


In [14]:
rf_scores = cross_validate(
    random_forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Random Forest Results")
print("Mean ROC-AUC:", rf_scores["test_roc_auc"].mean())
print("Mean F1:", rf_scores["test_f1"].mean())

Random Forest Results
Mean ROC-AUC: 0.8013836602735882
Mean F1: 0.4644183079447618


In [15]:
results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "CV ROC-AUC": logistic_scores["test_roc_auc"].mean(),
        "CV F1": logistic_scores["test_f1"].mean()
    },
    {
        "Model": "Random Forest",
        "CV ROC-AUC": rf_scores["test_roc_auc"].mean(),
        "CV F1": rf_scores["test_f1"].mean()
    }
])

print(results.round(3))

                 Model  CV ROC-AUC  CV F1
0  Logistic Regression       0.830  0.507
1        Random Forest       0.801  0.464


In [16]:
dummy_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy_scores = cross_validate(
    dummy_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

dummy_result = pd.DataFrame([{
    "Model": "Dummy Classifier",
    "CV ROC-AUC": dummy_scores["test_roc_auc"].mean(),
    "CV F1": dummy_scores["test_f1"].mean()
}])

results = pd.concat([results, dummy_result], ignore_index=True)

print(results.round(3))

                 Model  CV ROC-AUC  CV F1
0  Logistic Regression       0.830  0.507
1        Random Forest       0.801  0.464
2     Dummy Classifier       0.500  0.000


In [17]:
dummy_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy_scores = cross_validate(
    dummy_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

dummy_result = pd.DataFrame([{
    "Model": "Dummy Classifier",
    "CV ROC-AUC": dummy_scores["test_roc_auc"].mean(),
    "CV F1": dummy_scores["test_f1"].mean()
}])

results = pd.concat([results, dummy_result], ignore_index=True)

print(results.round(3))

                 Model  CV ROC-AUC  CV F1
0  Logistic Regression       0.830  0.507
1        Random Forest       0.801  0.464
2     Dummy Classifier       0.500  0.000
3     Dummy Classifier       0.500  0.000


In [18]:
results.to_csv(
    PROJECT_ROOT / "reports" / "results.csv",
    index=False
)

print("results.csv created successfully!")

results.csv created successfully!
